# Gated Recurrent Unit (GRU) - PyTorch

In [1]:
import os
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# -----------------------------
# Reproducibility: Set seeds
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------
# Device setup
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Generate synthetic dataset
# -----------------------------
samples, seq_len, features = 1400, 30, 3
X = np.random.normal(size=(samples, seq_len, features)).astype("float32")

signal = (
    X[:, -10:, 0].mean(axis=1) +
    0.5 * X[:, :10, 1].mean(axis=1)
)
y = (signal > 0.05).astype("int64")

# Train/test split
train_loader = DataLoader(
    TensorDataset(torch.tensor(X[:1000]), torch.tensor(y[:1000])),
    batch_size=64,
    shuffle=True,
)
test_x = torch.tensor(X[1000:], device=device)
test_y = torch.tensor(y[1000:], device=device)

# -----------------------------
# Define GRU model
# -----------------------------
class GRUClassifier(nn.Module):
    def __init__(self, input_dim: int = 3, hidden_dim: int = 32):
        super().__init__()
        self.recurrent = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=False,
        )
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        output, _ = self.recurrent(x)
        return self.classifier(output[:, -1, :])

# -----------------------------
# Training setup
# -----------------------------
model = GRUClassifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# -----------------------------
# Training loop
# -----------------------------
for epoch in range(10):
    model.train()
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        accuracy = (model(test_x).argmax(dim=1) == test_y).float().mean().item()
    print(f"epoch={epoch+1:02d} accuracy={accuracy:.3f}")

Using device: cuda
epoch=01 accuracy=0.625
epoch=02 accuracy=0.615
epoch=03 accuracy=0.710
epoch=04 accuracy=0.782
epoch=05 accuracy=0.782
epoch=06 accuracy=0.837
epoch=07 accuracy=0.832
epoch=08 accuracy=0.840
epoch=09 accuracy=0.822
epoch=10 accuracy=0.825


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------------
# 1. Device Configuration
# -----------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------------
# 2. Load Dataset
# -----------------------------------
VOCAB_SIZE = 10000
MAX_LEN = 200
BATCH_SIZE = 64

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

# Padding sequences
x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post')
x_test = pad_sequences(x_test, maxlen=MAX_LEN, padding='post')

# Convert to tensors
x_train = torch.tensor(x_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)

x_test = torch.tensor(x_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.float32)

# Create DataLoaders
train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# -----------------------------------
# 3. Define GRU Model
# -----------------------------------
class GRUNet(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super(GRUNet, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc1 = nn.Linear(hidden_dim, 32)
        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(32, output_dim)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        embedded = self.embedding(x)

        output, hidden = self.gru(embedded)

        # hidden shape:
        # (num_layers, batch_size, hidden_dim)

        hidden = hidden[-1]

        x = self.fc1(hidden)
        x = self.relu(x)

        x = self.fc2(x)

        return self.sigmoid(x)

# -----------------------------------
# 4. Initialize Model
# -----------------------------------
model = GRUNet(vocab_size=VOCAB_SIZE,
               embed_dim=128,
               hidden_dim=64,
               output_dim=1).to(device)

# -----------------------------------
# 5. Loss and Optimizer
# -----------------------------------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------------
# 6. Training Loop
# -----------------------------------
EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        labels = labels.unsqueeze(1)

        # Forward pass
        outputs = model(inputs)

        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

# -----------------------------------
# 7. Evaluation
# -----------------------------------
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        predictions = (outputs >= 0.5).float()

        total += labels.size(0)

        correct += (predictions.squeeze() == labels).sum().item()

accuracy = 100 * correct / total

print(f"\nTest Accuracy: {accuracy:.2f}%")

# -----------------------------------
# 8. Prediction Example
# -----------------------------------
sample = x_test[0].unsqueeze(0).to(device)

prediction = model(sample)

if prediction.item() > 0.5:
    print("Positive Review")
else:
    print("Negative Review")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch [1/5], Loss: 0.6573
Epoch [2/5], Loss: 0.4635
Epoch [3/5], Loss: 0.3052
Epoch [4/5], Loss: 0.2192
Epoch [5/5], Loss: 0.1650

Test Accuracy: 86.95%
Negative Review
